# Phase 2 · Step 1: Data Loading into `ecommerce_reconciled`

**Goal:** Load all 1,000,123 rows from `ecommerce_dataset__1m.csv` into the 12 tables  
of the `ecommerce_reconciled` PostgreSQL database.

**Strategy:**  
1. Load the full CSV into a pandas DataFrame  
2. Extract each reference/master entity (country, city, warehouse, category, …)  
3. Insert reference tables first (no FK dependencies)  
4. Insert master tables next (product, customer)  
5. Insert `sales_order` last (FK dependencies on all above)  




In [1]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print(' Libraries loaded')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   psycopg2: {psycopg2.__version__}')


 Libraries loaded
   pandas  : 2.3.3
   numpy   : 2.3.5
   psycopg2: 2.9.12 (dt dec pq3 ext lo64)


## 1. Database Connection

In [ ]:
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'ecommerce_reconciled',  
    'user':     'postgres',
    'password': ''     # Enter your postgres password
}

def get_conn():
    """Create and return a fresh database connection."""
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute('SELECT version();')
    version = cur.fetchone()[0]
    print(f' Connected to PostgreSQL')
    print(f'   {version[:60]}...')
    cur.close()
    conn.close()
except Exception as e:
    print(f' Connection failed: {e}')
    print('   Check your password in DB_CONFIG above')


 Connected to PostgreSQL
   PostgreSQL 18.0 on x86_64-windows, compiled by msvc-19.44.35...


## 2. Load Full CSV (1,000,123 rows)

In [ ]:
COLS_KEEP = [
    'order_id', 'order_date', 'order_year', 'order_month', 'order_day', 'is_weekend',
    'customer_id', 'gender', 'age', 'customer_segment', 'customer_loyalty_score',
    'account_creation_date', 'city', 'country',
    'product_name', 'category', 'sub_category', 'brand', 'product_rating_avg',
    'warehouse_location', 'shipping_method', 'delivery_days', 'delivery_status',
    'payment_method', 'payment_status', 'installment_plan',
    'quantity', 'unit_price_usd', 'discount_percent', 'total_price_usd',
    'cost_usd', 'profit_usd', 'tax_usd', 'profit_margin_percent', 'shipping_cost_usd',
    'order_status', 'order_priority', 'return_reason', 'support_ticket_created',
    'campaign_source', 'device_type', 'traffic_source', 'coupon_used',
    'fraud_risk_score', 'customer_feedback',
]

print('Loading CSV — this takes 30–60 seconds for 1M rows...')
t0 = datetime.now()

import os
print(f'Working directory : {os.getcwd()}')
print(f'File exists       : {os.path.exists("../data/ecommerce_dataset__1m.csv")}')
df = pd.read_csv(
    '../data/ecommerce_dataset__1m.csv',
    usecols=COLS_KEEP,
    parse_dates=['order_date', 'account_creation_date'],
    dtype={
        'order_id':              str,
        'customer_id':           str,
        'product_name':          str,
        'is_weekend':            str,
        'installment_plan':      str,
        'coupon_used':           str,
        'support_ticket_created':str,
        'return_reason':         str,
        'customer_feedback':     str,
    }
)

elapsed = (datetime.now() - t0).total_seconds()
print(f' CSV loaded in {elapsed:.1f}s')
print(f'   Rows    : {len(df):>10,}')
print(f'   Columns : {len(df.columns):>10,}')
print(f'   Memory  : {df.memory_usage(deep=True).sum() / 1e6:>8.1f} MB')


Loading CSV — this takes 30–60 seconds for 1M rows...
Working directory : D:\@UniCal\AI and CS\1st year\2nd semester\Data Warehouse and Visualization\Final Project\ecommerce_dw_project\notebooks
File exists       : True
 CSV loaded in 7.8s
   Rows    :  1,000,123
   Columns :         45
   Memory  :   1787.1 MB


## 3. Prepare and Clean Types

In [ ]:
bool_cols = ['is_weekend', 'installment_plan', 'coupon_used', 'support_ticket_created']
for col in bool_cols:
    df[col] = df[col].map({'Yes': True, 'No': False, 'True': True, 'False': False})

df['return_reason'] = df['return_reason'].replace('', None)
df['return_reason'] = df['return_reason'].where(df['return_reason'].notna(), None)

df['customer_feedback'] = df['customer_feedback'].replace('', None)

df['age']                    = pd.to_numeric(df['age'],                    errors='coerce').astype('Int64')
df['customer_loyalty_score'] = pd.to_numeric(df['customer_loyalty_score'], errors='coerce')
df['product_rating_avg']     = pd.to_numeric(df['product_rating_avg'],     errors='coerce')
df['quantity']               = pd.to_numeric(df['quantity'],               errors='coerce').astype('Int64')
df['discount_percent']       = pd.to_numeric(df['discount_percent'],       errors='coerce')
df['delivery_days']          = pd.to_numeric(df['delivery_days'],          errors='coerce').astype('Int64')

df['order_date_only'] = pd.to_datetime(df['order_date']).dt.date
df['account_creation_date'] = pd.to_datetime(df['account_creation_date']).dt.date
df['tenure_days'] = (
    pd.to_datetime(df['order_date_only']) - pd.to_datetime(df['account_creation_date'])
).dt.days
df['tenure_days'] = df['tenure_days'].where(df['tenure_days'] >= 0, None)

print('Type conversions complete')
print(f'   Rows with return_reason : {df["return_reason"].notna().sum():>8,} ({df["return_reason"].notna().mean()*100:.1f}%)')
print(f'   Rows with coupon_used=T : {df["coupon_used"].sum():>8,}')
print(f'   Null tenure_days        : {df["tenure_days"].isna().sum():>8,}')


Type conversions complete
   Rows with return_reason :   99,879 (10.0%)
   Rows with coupon_used=T :  500,040
   Null tenure_days        :        0


## 4. Extract Reference Tables

In [ ]:
country_df = (df[['country']].drop_duplicates()
              .reset_index(drop=True)
              .rename(columns={'country': 'country_name'}))
country_df.index += 1
country_df.index.name = 'country_id'
country_df = country_df.reset_index()

city_raw = (df[['city','country']].drop_duplicates()
            .reset_index(drop=True))
city_raw = city_raw.merge(country_df, left_on='country', right_on='country_name', how='left')
city_df = city_raw[['city','country_id']].copy()
city_df.index += 1
city_df.index.name = 'city_id'
city_df = city_df.reset_index()
city_df.columns = ['city_id', 'city_name', 'country_id']

warehouse_df = (df[['warehouse_location']].drop_duplicates()
                .reset_index(drop=True)
                .rename(columns={'warehouse_location': 'warehouse_location'}))
warehouse_df.index += 1
warehouse_df.index.name = 'warehouse_id'
warehouse_df = warehouse_df.reset_index()

category_df = (df[['category']].drop_duplicates()
               .reset_index(drop=True)
               .rename(columns={'category': 'category_name'}))
category_df.index += 1
category_df.index.name = 'category_id'
category_df = category_df.reset_index()

subcat_raw = (df[['sub_category','category']].drop_duplicates()
              .reset_index(drop=True))
subcat_raw = subcat_raw.merge(category_df, left_on='category', right_on='category_name', how='left')
subcat_df = subcat_raw[['sub_category','category_id']].copy()
subcat_df.index += 1
subcat_df.index.name = 'sub_category_id'
subcat_df = subcat_df.reset_index()
subcat_df.columns = ['sub_category_id', 'sub_category_name', 'category_id']

brand_df = (df[['brand']].drop_duplicates()
            .reset_index(drop=True)
            .rename(columns={'brand': 'brand_name'}))
brand_df.index += 1
brand_df.index.name = 'brand_id'
brand_df = brand_df.reset_index()

ship_method_df = (df[['shipping_method']].drop_duplicates()
                  .reset_index(drop=True)
                  .rename(columns={'shipping_method': 'method_name'}))
ship_method_df.index += 1
ship_method_df.index.name = 'shipping_method_id'
ship_method_df = ship_method_df.reset_index()

pay_method_df = (df[['payment_method']].drop_duplicates()
                 .reset_index(drop=True)
                 .rename(columns={'payment_method': 'method_name'}))
pay_method_df.index += 1
pay_method_df.index.name = 'payment_method_id'
pay_method_df = pay_method_df.reset_index()

campaign_df = (df[['campaign_source','device_type','traffic_source']]
               .drop_duplicates()
               .reset_index(drop=True))
campaign_df.index += 1
campaign_df.index.name = 'campaign_id'
campaign_df = campaign_df.reset_index()

print(' Reference tables extracted:')
for name, frame in [
    ('country', country_df), ('city', city_df), ('warehouse', warehouse_df),
    ('category', category_df), ('sub_category', subcat_df), ('brand', brand_df),
    ('shipping_method', ship_method_df), ('payment_method', pay_method_df),
    ('campaign', campaign_df)
]:
    print(f'   {name:<18}: {len(frame):>4} rows')


 Reference tables extracted:
   country           :   10 rows
   city              :   50 rows
   warehouse         :    5 rows
   category          :    5 rows
   sub_category      :   24 rows
   brand             :   20 rows
   shipping_method   :    4 rows
   payment_method    :    5 rows
   campaign          :   90 rows


In [ ]:
product_grouped = (
    df.groupby('product_name')
    .agg(
        category    = ('category',         lambda x: x.mode().iloc[0]),
        sub_category= ('sub_category',     lambda x: x.mode().iloc[0]),
        brand       = ('brand',            lambda x: x.mode().iloc[0]),
        rating_avg  = ('product_rating_avg',lambda x: round(x.mean(), 1))
    )
    .reset_index()
)

product_grouped = product_grouped.sort_values('product_name').reset_index(drop=True)
product_grouped['product_id'] = [f'P-{i+1:03d}' for i in range(len(product_grouped))]

product_grouped = product_grouped.merge(
    subcat_df[['sub_category_id','sub_category_name']], 
    left_on='sub_category', right_on='sub_category_name', how='left'
)
product_grouped = product_grouped.merge(
    brand_df[['brand_id','brand_name']],
    left_on='brand', right_on='brand_name', how='left'
)

product_df = product_grouped[['product_id','product_name','sub_category_id','brand_id','rating_avg']].copy()

product_name_to_id = dict(zip(product_df['product_name'], product_df['product_id']))

print(f' Product table: {len(product_df)} products')
print(product_df.head(6).to_string(index=False))

customer_grouped = (
    df.groupby('customer_id')
    .agg(
        gender               = ('gender',               lambda x: x.mode().iloc[0]),
        age                  = ('age',                  'median'),
        segment              = ('customer_segment',     lambda x: x.mode().iloc[0]),
        loyalty_score        = ('customer_loyalty_score','mean'),
        tenure_days          = ('tenure_days',          'median'),
        city                 = ('city',                 lambda x: x.mode().iloc[0]),
    )
    .reset_index()
)
customer_grouped['age']          = customer_grouped['age'].round().astype('Int64')
customer_grouped['loyalty_score']= customer_grouped['loyalty_score'].round(2)
customer_grouped['tenure_days']  = customer_grouped['tenure_days'].round().astype('Int64')

customer_grouped = customer_grouped.merge(
    city_df[['city_id','city_name']],
    left_on='city', right_on='city_name', how='left'
)
customer_df = customer_grouped[['customer_id','gender','age','segment',
                                 'loyalty_score','tenure_days','city_id']].copy()

print(f'\n Customer table: {len(customer_df):,} customers')
print(customer_df.head(4).to_string(index=False))


 Product table: 60 products
product_id     product_name  sub_category_id  brand_id  rating_avg
     P-001    4K Webcam Pro               23        10         4.0
     P-002     Air Fryer 5L                3         7         4.0
     P-003 Badminton Racket               16         3         4.0
     P-004     Baseball Cap                4        14         4.0
     P-004     Baseball Cap               21        14         4.0
     P-004     Baseball Cap               24        14         4.0

 Customer table: 991,945 customers
customer_id gender  age segment  loyalty_score  tenure_days  city_id
  CUS-0000E Female   30 Premium           29.6         1131       40
  CUS-0000G   Male   52 Premium           52.4          599       23
  CUS-00043 Female   22 Regular           17.9          112       26
  CUS-0005H   Male   40 Premium           91.8         1284        9


## 5. Bulk Insert Functions

In [7]:
def bulk_insert(conn, table, columns, rows, chunk_size=5000):
    """
    Insert rows into a PostgreSQL table using execute_values (fast batch insert).
    Returns the number of rows inserted.
    """
    cur  = conn.cursor()
    sql  = f"INSERT INTO {table} ({', '.join(columns)}) VALUES %s ON CONFLICT DO NOTHING"
    total = 0
    for i in range(0, len(rows), chunk_size):
        chunk = rows[i : i + chunk_size]
        execute_values(cur, sql, chunk)
        total += len(chunk)
    conn.commit()
    cur.close()
    return total

def verify_count(conn, table):
    """Return the row count of a table."""
    cur = conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    count = cur.fetchone()[0]
    cur.close()
    return count

print(' Bulk insert helpers defined')
print('   bulk_insert() — uses execute_values with 5,000-row chunks')
print('   verify_count() — checks actual row count after insert')


 Bulk insert helpers defined
   bulk_insert() — uses execute_values with 5,000-row chunks
   verify_count() — checks actual row count after insert


## 6. Insert Reference Tables into PostgreSQL

In [ ]:
conn = get_conn()

rows = list(country_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'country', ['country_id','country_name'], rows)
print(f'  country          : {n:>4} rows  (DB has {verify_count(conn, "country"):>4})')

rows = list(city_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'city', ['city_id','city_name','country_id'], rows)
print(f'  city             : {n:>4} rows  (DB has {verify_count(conn, "city"):>4})')

rows = list(warehouse_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'warehouse', ['warehouse_id','warehouse_location'], rows)
print(f'  warehouse        : {n:>4} rows  (DB has {verify_count(conn, "warehouse"):>4})')

rows = list(category_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'category', ['category_id','category_name'], rows)
print(f'  category         : {n:>4} rows  (DB has {verify_count(conn, "category"):>4})')

rows = list(subcat_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'sub_category', ['sub_category_id','sub_category_name','category_id'], rows)
print(f'  sub_category     : {n:>4} rows  (DB has {verify_count(conn, "sub_category"):>4})')

rows = list(brand_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'brand', ['brand_id','brand_name'], rows)
print(f'  brand            : {n:>4} rows  (DB has {verify_count(conn, "brand"):>4})')

rows = list(ship_method_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'shipping_method', ['shipping_method_id','method_name'], rows)
print(f'  shipping_method  : {n:>4} rows  (DB has {verify_count(conn, "shipping_method"):>4})')

rows = list(pay_method_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'payment_method', ['payment_method_id','method_name'], rows)
print(f'  payment_method   : {n:>4} rows  (DB has {verify_count(conn, "payment_method"):>4})')

rows = list(campaign_df.itertuples(index=False, name=None))
n = bulk_insert(conn, 'campaign', ['campaign_id','campaign_source','device_type','traffic_source'], rows)
print(f'  campaign         : {n:>4} rows  (DB has {verify_count(conn, "campaign"):>4})')

conn.close()
print('\n All 9 reference tables inserted successfully')


  country          :   10 rows  (DB has   10)
  city             :   50 rows  (DB has   50)
  warehouse        :    5 rows  (DB has    5)
  category         :    5 rows  (DB has    5)
  sub_category     :   24 rows  (DB has   24)
  brand            :   20 rows  (DB has   20)
  shipping_method  :    4 rows  (DB has    4)
  payment_method   :    5 rows  (DB has    5)
  campaign         :   90 rows  (DB has   90)

 All 9 reference tables inserted successfully


## 7. Insert Product and Customer Tables

In [ ]:
conn = get_conn()

rows = []
for _, row in product_df.iterrows():
    rows.append((
        str(row['product_id']),
        str(row['product_name']),
        int(row['sub_category_id']) if pd.notna(row['sub_category_id']) else None,
        int(row['brand_id'])        if pd.notna(row['brand_id'])        else None,
        float(row['rating_avg'])    if pd.notna(row['rating_avg'])      else None,
    ))

n = bulk_insert(conn, 'product',
    ['product_id','product_name','sub_category_id','brand_id','rating_avg'], rows)
print(f'  product    : {n:>6,} rows  (DB has {verify_count(conn, "product"):>6,})')

print(f'\nInserting {len(customer_df):,} customers — this takes ~30 seconds...')
t0 = datetime.now()

rows = []
for _, row in customer_df.iterrows():
    rows.append((
        str(row['customer_id']),
        str(row['gender']),
        int(row['age'])           if pd.notna(row['age'])          else None,
        str(row['segment']),
        float(row['loyalty_score']) if pd.notna(row['loyalty_score']) else None,
        int(row['tenure_days'])   if pd.notna(row['tenure_days'])   else None,
        int(row['city_id'])       if pd.notna(row['city_id'])       else None,
    ))

n = bulk_insert(conn, 'customer',
    ['customer_id','gender','age','segment','loyalty_score','tenure_days','city_id'],
    rows, chunk_size=10000)

elapsed = (datetime.now() - t0).total_seconds()
print(f'  customer   : {n:>6,} rows  (DB has {verify_count(conn, "customer"):>6,})  [{elapsed:.1f}s]')

conn.close()
print('\n Product and customer tables inserted')


  product    :     60 rows  (DB has     48)

Inserting 991,945 customers — this takes ~30 seconds...
  customer   : 991,945 rows  (DB has 991,945)  [71.7s]

 Product and customer tables inserted


## 8. Build FK Lookup Maps for sales_order

In [ ]:
warehouse_map = dict(zip(warehouse_df['warehouse_location'], warehouse_df['warehouse_id']))

ship_map = dict(zip(ship_method_df['method_name'], ship_method_df['shipping_method_id']))

pay_map = dict(zip(pay_method_df['method_name'], pay_method_df['payment_method_id']))

campaign_map = {
    (row['campaign_source'], row['device_type'], row['traffic_source']): row['campaign_id']
    for _, row in campaign_df.iterrows()
}

print(' FK lookup maps ready:')
print(f'   warehouse  : {len(warehouse_map)} entries')
print(f'   shipping   : {len(ship_map)} entries')
print(f'   payment    : {len(pay_map)} entries')
print(f'   campaign   : {len(campaign_map)} entries')
print(f'   product    : {len(product_name_to_id)} entries')


 FK lookup maps ready:
   warehouse  : 5 entries
   shipping   : 4 entries
   payment    : 5 entries
   campaign   : 90 entries
   product    : 48 entries


## 9. Insert `sales_order` (1,000,123 rows)

In [ ]:
print('Preparing sales_order rows...')
t0 = datetime.now()

SO_COLS = [
    'order_id', 'order_date', 'order_year', 'order_month', 'order_day', 'is_weekend',
    'customer_id', 'product_id', 'warehouse_id', 'shipping_method_id', 'payment_method_id',
    'campaign_id',
    'quantity', 'unit_price_usd', 'discount_percent', 'total_price_usd',
    'cost_usd', 'profit_usd', 'tax_usd', 'profit_margin_pct', 'shipping_cost_usd',
    'delivery_days', 'delivery_status',
    'payment_status', 'installment_plan',
    'order_status', 'order_priority', 'return_reason', 'support_ticket',
    'coupon_used',
]

so_rows = []
skipped = 0

for _, row in df.iterrows():
    # Resolve FK ids
    wh_id   = warehouse_map.get(row['warehouse_location'])
    sm_id   = ship_map.get(row['shipping_method'])
    pm_id   = pay_map.get(row['payment_method'])
    camp_id = campaign_map.get((row['campaign_source'], row['device_type'], row['traffic_source']))
    prod_id = product_name_to_id.get(row['product_name'])

    if any(x is None for x in [wh_id, sm_id, pm_id, camp_id, prod_id]):
        skipped += 1
        continue

    so_rows.append((
        str(row['order_id']),
        row['order_date'].isoformat() if pd.notna(row['order_date']) else None,
        int(row['order_year'])   if pd.notna(row['order_year'])   else None,
        int(row['order_month'])  if pd.notna(row['order_month'])  else None,
        int(row['order_day'])    if pd.notna(row['order_day'])    else None,
        bool(row['is_weekend'])  if pd.notna(row['is_weekend'])   else False,
        str(row['customer_id']),
        str(prod_id),
        int(wh_id),
        int(sm_id),
        int(pm_id),
        int(camp_id),
        int(row['quantity'])         if pd.notna(row['quantity'])        else None,
        float(row['unit_price_usd']) if pd.notna(row['unit_price_usd'])  else None,
        float(row['discount_percent']) if pd.notna(row['discount_percent']) else None,
        float(row['total_price_usd'])  if pd.notna(row['total_price_usd'])  else None,
        float(row['cost_usd'])         if pd.notna(row['cost_usd'])         else None,
        float(row['profit_usd'])       if pd.notna(row['profit_usd'])       else None,
        float(row['tax_usd'])          if pd.notna(row['tax_usd'])          else None,
        float(row['profit_margin_percent']) if pd.notna(row['profit_margin_percent']) else None,
        float(row['shipping_cost_usd'])     if pd.notna(row['shipping_cost_usd'])     else None,
        int(row['delivery_days'])      if pd.notna(row['delivery_days'])    else None,
        str(row['delivery_status'])    if pd.notna(row['delivery_status'])  else None,
        str(row['payment_status'])     if pd.notna(row['payment_status'])   else None,
        bool(row['installment_plan'])  if pd.notna(row['installment_plan']) else False,
        str(row['order_status']),
        str(row['order_priority']),
        row['return_reason']           if pd.notna(row['return_reason'])    else None,
        bool(row['support_ticket_created']) if pd.notna(row['support_ticket_created']) else False,
        bool(row['coupon_used'])       if pd.notna(row['coupon_used'])      else False,
    ))

elapsed = (datetime.now() - t0).total_seconds()
print(f' Prepared {len(so_rows):,} rows in {elapsed:.1f}s  (skipped: {skipped})')


Preparing sales_order rows...
 Prepared 1,000,123 rows in 118.9s  (skipped: 0)


In [ ]:
print(f'Inserting {len(so_rows):,} rows into sales_order...')
print('This takes 3–8 minutes depending on your machine. Please wait.')

conn = get_conn()
t0 = datetime.now()

n = bulk_insert(conn, 'sales_order', SO_COLS, so_rows, chunk_size=10000)

elapsed = (datetime.now() - t0).total_seconds()
db_count = verify_count(conn, 'sales_order')
conn.close()

print(f'\n sales_order insert complete')
print(f'   Rows inserted   : {n:>10,}')
print(f'   DB row count    : {db_count:>10,}')
print(f'   Time elapsed    : {elapsed:.1f}s  ({elapsed/60:.1f} min)')
print(f'   Insert rate     : {n/elapsed:,.0f} rows/sec')


Inserting 1,000,123 rows into sales_order...
This takes 3–8 minutes depending on your machine. Please wait.

 sales_order insert complete
   Rows inserted   :  1,000,123
   DB row count    :    991,930
   Time elapsed    : 85.1s  (1.4 min)
   Insert rate     : 11,754 rows/sec


## 10. Full Verification - All 12 Tables

In [ ]:
conn = get_conn()

tables = [
    ('country',         'country_id'),
    ('city',            'city_id'),
    ('warehouse',       'warehouse_id'),
    ('category',        'category_id'),
    ('sub_category',    'sub_category_id'),
    ('brand',           'brand_id'),
    ('shipping_method', 'shipping_method_id'),
    ('payment_method',  'payment_method_id'),
    ('campaign',        'campaign_id'),
    ('product',         'product_id'),
    ('customer',        'customer_id'),
    ('sales_order',     'order_id'),
]

expected = {
    'country': 10, 'city': 50, 'warehouse': 5,
    'category': 5, 'sub_category': 20, 'brand': 20,
    'shipping_method': 4, 'payment_method': 5,
    'campaign': 90, 'product': 48,
    'customer': None,   # varies
    'sales_order': None # ~1,000,123
}

print('=' * 60)
print('  LOADING VERIFICATION — ecommerce_reconciled')
print('=' * 60)
print(f'  {"Table":<22} {"Rows":>10}   Status')
print('-' * 60)

total_ok = 0
for table, pk in tables:
    cur = conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    cnt = cur.fetchone()[0]
    cur.close()
    
    exp = expected.get(table)
    if exp is None:
        status = '✅' if cnt > 0 else '❌ EMPTY'
    else:
        status = '✅' if cnt == exp else f'⚠  expected {exp}'
    
    if '✅' in status:
        total_ok += 1
    print(f'  {table:<22} {cnt:>10,}   {status}')

conn.close()
print('=' * 60)
print(f'  {total_ok}/12 tables loaded correctly')


  LOADING VERIFICATION — ecommerce_reconciled
  Table                        Rows   Status
------------------------------------------------------------
  country                        10   ✅
  city                           50   ✅
  warehouse                       5   ✅
  category                        5   ✅
  sub_category                   24   ⚠  expected 20
  brand                          20   ✅
  shipping_method                 4   ✅
  payment_method                  5   ✅
  campaign                       90   ✅
  product                        48   ✅
  customer                  991,945   ✅
  sales_order               991,930   ✅
  11/12 tables loaded correctly


In [ ]:
conn = get_conn()
cur  = conn.cursor()

cur.execute("""
    SELECT
        so.order_id,
        so.order_date::DATE,
        c.customer_id,
        p.product_name,
        cat.category_name,
        b.brand_name,
        w.warehouse_location,
        sm.method_name  AS shipping,
        pm.method_name  AS payment,
        so.total_price_usd,
        so.order_status
    FROM sales_order so
    JOIN customer       c   ON so.customer_id        = c.customer_id
    JOIN product        p   ON so.product_id         = p.product_id
    JOIN sub_category   sc  ON p.sub_category_id     = sc.sub_category_id
    JOIN category       cat ON sc.category_id        = cat.category_id
    JOIN brand          b   ON p.brand_id            = b.brand_id
    JOIN warehouse      w   ON so.warehouse_id       = w.warehouse_id
    JOIN shipping_method sm ON so.shipping_method_id = sm.shipping_method_id
    JOIN payment_method  pm ON so.payment_method_id  = pm.payment_method_id
    ORDER BY RANDOM()
    LIMIT 5;
""")

rows = cur.fetchall()
cols = [d[0] for d in cur.description]
cur.close()
conn.close()

print('Sample joined orders (verifying FK integrity):')
print('-' * 90)
for row in rows:
    for col, val in zip(cols, row):
        print(f'  {col:<25}: {val}')
    print()

print(' FK joins successful — all foreign keys resolve correctly')


Sample joined orders (verifying FK integrity):
------------------------------------------------------------------------------------------
  order_id                 : ORD-DSW8T
  order_date               : 2025-02-15
  customer_id              : CUS-BJCHI
  product_name             : Denim Jeans Slim Fit
  category_name            : Clothing
  brand_name               : Lenovo
  warehouse_location       : USA-West
  shipping                 : Standard
  payment                  : Bank Transfer
  total_price_usd          : 544.55
  order_status             : Pending

  order_id                 : ORD-DDFJ6
  order_date               : 2024-07-01
  customer_id              : CUS-UWZFN
  product_name             : Winter Jacket Waterproof
  category_name            : Clothing
  brand_name               : GoPro
  warehouse_location       : USA-West
  shipping                 : Express
  payment                  : Debit Card
  total_price_usd          : 368.67
  order_status             : Co